<div style="font-family: Arial, sans-serif; background-color: #f8f9fa; padding: 20px; border-radius: 10px;">
  <img src="https://sigmoidal.ai/wp-content/uploads/2024/09/Academia-Sigmoidal-Light.png" alt="Academia Sigmoidal" width="250">
  <h1 style="color: #007bff; font-size: 24px;">Bootcamp de Visão Computacional com Drones</h1>
  <h3 style="color: #343a40; font-size: 20px;">Projeto 4: Quantos bovinos aparecem nesta imagem?</h3>
  <p style="color: #6c757d; font-size: 14px;"><strong>Instrutor:</strong> Carlos Melo, MSc.</p>
</div>


# Quantos bovinos aparecem nesta imagem?

Vamos auditar os dados antes de usar um detector. O objetivo é entender as anotações, a escala dos animais e a separação entre desenvolvimento e teste.

Este notebook é independente: os dados necessários acompanham o repositório. No Colab, selecione **Copiar para o Drive** e execute as células na ordem. A CPU é suficiente para esta exploração.

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlosfab/bootcamp-visao-computacional-drones/blob/codex/projeto-4-gado/projeto-4/00_dados_e_problema.ipynb)


## Ambiente

A primeira célula obtém os arquivos da versão do projeto e instala as dependências. O PyTorch fornecido pelo ambiente será preservado; sua versão será registrada. Se já importou bibliotecas antes desta instalação, reinicie o ambiente e execute novamente desde o início.

Os arquivos de trabalho ficam em `dados/`, `pesos/` e `resultados/`. As pastas de métricas e figuras têm nomes fixos e seus arquivos são atualizados ao reexecutar. Antes de repetir o experimento ou encerrar o Colab, baixe o ZIP seguindo as instruções ao final.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
if not Path("suporte").is_dir():
    destino = Path("/content/bootcamp-gado")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "--branch", "codex/projeto-4-gado",
                        "https://github.com/carlosfab/bootcamp-visao-computacional-drones.git", str(destino)], check=True)
    subprocess.run(["git", "-C", str(destino), "sparse-checkout", "set", "projeto-4"], check=True)
    os.chdir(destino / "projeto-4")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


In [ ]:
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from suporte.preparar_dados import obter_dados
from suporte.experimento import inventario, ler_caixas, desenhar
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 12,
                     "font.family": "sans-serif", "axes.grid": False})


A semente controla parte da aleatoriedade, mas versões, hardware e operações numéricas também influenciam o resultado. O registro abaixo identifica o ambiente desta execução, sem prometer identidade bit a bit entre CPU e GPU.


In [ ]:
from importlib.metadata import version


In [ ]:
SAIDA = Path("resultados/dados")
SAIDA.mkdir(parents=True, exist_ok=True)
pacotes = ["numpy", "pandas", "matplotlib", "Pillow", "ultralytics", "torch"]
ambiente = {nome: version(nome) for nome in pacotes}
ambiente["python"] = platform.python_version()
ambiente["sistema"] = platform.platform()

(SAIDA / "ambiente.json").write_text(json.dumps(ambiente, indent=2))
display(ambiente)


## Dados e partições

O recorte didático contém **300 imagens** de bovinos a pasto: 180 de treino, 60 de validação e 60 de teste. As imagens derivam do ICAERUS v2, com lado maior limitado a 2048 pixels e anotações YOLO de uma classe, `cow`. A atribuição e a licença CC BY 4.0 acompanham os dados.

Treino e validação usam Mauron e Jalogny em datas separadas. O teste usa Derval, uma fazenda ausente do desenvolvimento. Datas e voos não atravessam partições, mas os animais não têm identificadores individuais. A avaliação mede contagem **por imagem**, não um censo de indivíduos únicos.


In [ ]:
RAIZ = obter_dados()
tabela = inventario(RAIZ)
manifesto = json.loads((RAIZ / "manifesto.json").read_text())
treino = tabela[tabela["split"] == "train"].copy()
validacao = tabela[tabela["split"] == "val"].copy()
teste = tabela[tabela["split"] == "test"].copy()
assert [len(treino), len(validacao), len(teste)] == [180, 60, 60]
print("Imagens de treino, validação e teste:", len(treino), len(validacao), len(teste))


## O teste permanece reservado

O inventário verifica que cada imagem tem seu arquivo de anotação. Para explorar distribuições, selecionar exemplos e formular hipóteses, usaremos somente **treino e validação**. As caixas de teste serão usadas posteriormente, com as configurações já fixadas.

Um arquivo de anotação vazio significa imagem negativa. Um arquivo ausente é erro de integridade e não deve virar silenciosamente uma contagem zero.


In [ ]:
desenvolvimento = tabela[tabela["split"].isin(["train", "val"])].copy()
resumo = desenvolvimento.groupby("split").agg(
    imagens=("image_id", "size"), caixas=("count", "sum"),
    negativas=("count", lambda x: int((x == 0).sum())))
display(resumo)
resumo.to_csv(SAIDA / "resumo_desenvolvimento.csv")


**Imagens negativas** avaliam a capacidade de não inventar animais onde não há bovinos anotados. O treino foi enriquecido com imagens positivas; as proporções de treino e validação não precisam coincidir. Reportar apenas uma média sobre todas as imagens pode esconder comportamento ruim no subconjunto positivo.


In [ ]:
for split, grupo in desenvolvimento.groupby("split"):
    plt.hist(grupo["count"], bins=np.arange(0, desenvolvimento["count"].max() + 2),
             alpha=0.6, label=split)
plt.xlabel("Bovinos anotados por imagem")
plt.ylabel("Imagens")
plt.legend()
plt.show()


## Separação por captura

Fotografias de um mesmo voo podem ser muito parecidas. Conferiremos os grupos registrados no manifesto e a política de partição, sem inspecionar as imagens do teste. Mesmo com datas diferentes, treino e validação podem mostrar os mesmos animais; o experimento não testa identificação individual.


In [ ]:
amostras = pd.DataFrame(manifesto["samples"])
display(amostras.groupby(["split", "farm"]).size().rename("imagens").to_frame())
grupos = amostras.groupby(["farm", "flight"])["split"].nunique()
assert grupos.max() == 1, "Um voo aparece em mais de uma partição."
assert set(amostras.loc[amostras["split"] == "test", "farm"]) == {"Derval"}
assert "Derval" not in set(amostras.loc[amostras["split"] != "test", "farm"])
print("Voos separados; Derval reservada para o teste.")


## Ler uma anotação

Cada linha YOLO contém `classe centro_x centro_y largura altura`, com coordenadas normalizadas. Multiplicar as coordenadas horizontais pela largura da imagem e as verticais pela altura recupera pixels. Para desenhar uma caixa, usamos seus cantos `x_min, y_min, x_max, y_max`.


In [ ]:
exemplo = treino[treino["count"] > 0].sort_values("image_id").iloc[0]
linhas = Path(exemplo["label"]).read_text().splitlines()
print("Primeira anotação:", linhas[0])
caixas = ler_caixas(exemplo["label"], int(exemplo["width"]), int(exemplo["height"]))
print("Primeira caixa em pixels:", caixas[0].round(1))
print("Total de caixas:", len(caixas))


In [ ]:
figura = desenhar(exemplo["image"], caixas, [], [], 0.25)
plt.title(f"{len(caixas)} bovinos anotados")
plt.gca().get_legend().remove()
figura.savefig(SAIDA / "referencia_anotada.png", dpi=120)
plt.show()
plt.close(figura)


As caixas são a **referência anotada**, que também pode conter ambiguidades. Registre casos suspeitos sem alterar silenciosamente os arquivos. Uma correção de rótulos deve produzir uma nova versão dos dados e uma nova avaliação das configurações comparadas.

## O tamanho do animal na entrada da rede

O tamanho equivalente de uma caixa é a raiz quadrada de sua área, em pixels. O inventário calcula a mediana desses tamanhos por imagem; não confunda essa estatística com a mediana de todas as caixas do conjunto.

Ao reduzir o lado maior da foto para 640 pixels, cada animal também encolhe. A estimativa abaixo considera o redimensionamento proporcional, antes do preenchimento usado pelo detector.


In [ ]:
desenvolvimento["fator_640"] = 640 / desenvolvimento[["width", "height"]].max(axis=1)
desenvolvimento["lado_640"] = desenvolvimento["median_box_side"] * desenvolvimento["fator_640"]
colunas = ["width", "height", "median_box_side", "lado_640"]
display(desenvolvimento[colunas].describe().round(2))
desenvolvimento.to_csv(SAIDA / "inventario_desenvolvimento.csv", index=False)


In [ ]:
plt.hist(desenvolvimento["median_box_side"].dropna(), bins=20, alpha=0.6, label="Imagem")
plt.hist(desenvolvimento["lado_640"].dropna(), bins=20, alpha=0.6, label="Entrada 640")
plt.xlabel("Tamanho mediano das caixas por imagem, em pixels")
plt.ylabel("Imagens")
plt.legend()
plt.show()


Para observar a perda de detalhe, selecionamos uma imagem positiva da validação com animais pequenos. Exibir uma versão reduzida em um painel grande apenas amplia os pixels restantes; não recupera detalhe descartado.


In [ ]:
pequeno = validacao[validacao["count"] > 0].sort_values("median_box_side").iloc[0]
foto = Image.open(pequeno["image"]).convert("RGB")
reduzida = foto.copy()
reduzida.thumbnail((640, 640))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for eixo, imagem, titulo in zip(axes, [foto, reduzida], ["Imagem disponível", "Lado maior 640"]):
    eixo.imshow(imagem)
    eixo.set_title(titulo)
    eixo.axis("off")
plt.show()


## Comparar a mesma região em detalhe

As duas fotografias inteiras ocupam quase o mesmo espaço na tela, o que pode esconder a perda de detalhe. Vamos ampliar a região do primeiro bovino anotado neste exemplo de validação, incluindo uma margem e respeitando os limites da imagem.

O recorte é definido na imagem disponível. Para localizar a mesma região na versão reduzida, multiplicamos suas coordenadas pela proporção entre as dimensões das duas imagens. Arredondamos os limites para cobrir pixels inteiros e mantemos o mesmo campo de visão nos dois painéis.


In [ ]:
caixa_zoom = ler_caixas(pequeno["label"], foto.width, foto.height)[0]
centro = (caixa_zoom[:2] + caixa_zoom[2:]) / 2
raio = max(32, 2 * float(np.max(caixa_zoom[2:] - caixa_zoom[:2])))
inicio_zoom = np.maximum(0, np.floor(centro - raio)).astype(int)
fim_zoom = np.minimum(foto.size, np.ceil(centro + raio)).astype(int)
escala = np.array(reduzida.size) / np.array(foto.size)
inicio_red = np.floor(inicio_zoom * escala).astype(int)
fim_red = np.minimum(reduzida.size, np.ceil(fim_zoom * escala)).astype(int)


In [ ]:
zoom_original = foto.crop((*inicio_zoom, *fim_zoom))
zoom_reduzido = reduzida.crop((*inicio_red, *fim_red))
campo_original = [inicio_zoom[0], fim_zoom[0], fim_zoom[1], inicio_zoom[1]]
campo_reduzido = [inicio_red[0] / escala[0], fim_red[0] / escala[0],
                 fim_red[1] / escala[1], inicio_red[1] / escala[1]]
print("Pixels no recorte disponível:", zoom_original.size)
print("Pixels no recorte reduzido:", zoom_reduzido.size)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
paineis = [(zoom_original, campo_original, "Imagem disponível"),
           (zoom_reduzido, campo_reduzido, "Após redução para 640")]
for eixo, (imagem, campo, titulo) in zip(axes, paineis):
    eixo.imshow(imagem, extent=campo, interpolation="nearest")
    eixo.set_xlim(inicio_zoom[0], fim_zoom[0])
    eixo.set_ylim(fim_zoom[1], inicio_zoom[1])
    eixo.set_title(titulo)
    eixo.axis("off")
fig.savefig(SAIDA / "zoom_resolucao.png", dpi=120)
plt.show()


A exibição usa o vizinho mais próximo para deixar os pixels visíveis, sem suavizar a ampliação. Os dois painéis mostram a mesma região, mas o recorte reduzido dispõe de menos pixels para representar o bovino. Aumentar o painel não recupera a informação descartada.


## Antes de treinar

Registre uma hipótese sobre as dificuldades do detector: tamanho aparente, sombra, fundo ou proximidade entre bovinos. Use exemplos de treino ou validação como evidência. Não atribua mudanças de escala à altitude sem metadados que sustentem essa interpretação.

No próximo notebook, verificaremos se um detector genérico já produz uma contagem útil. A hipótese deve poder ser rejeitada pelos resultados.


## Salvar a execução

A pasta de um treinamento concluído é protegida contra reutilização; uma nova tentativa exige outro `name`. Já as pastas de métricas, configurações e figuras têm nomes fixos e são atualizadas ao reexecutar os notebooks. Baixe os resultados **antes de repetir** ou encerrar o runtime.

No Colab, copie o trecho opcional abaixo para uma nova célula e execute-o quando quiser baixar a execução. O ZIP inclui `resultados/` inteiro, os pesos treinados que estiverem nessa pasta e o manifesto. O checkpoint de referência do notebook de recortes continua disponível nos arquivos do projeto e é identificado pelo hash da configuração.

```python
import shutil
from datetime import datetime
from google.colab import files
shutil.copy("assets/manifesto.json", "resultados/manifesto.json")
nome = "resultados-projeto4-" + datetime.now().strftime("%Y%m%d-%H%M%S")
arquivo = shutil.make_archive(nome, "zip", "resultados")
files.download(arquivo)
```

Salve também este notebook com suas saídas em **Arquivo > Salvar** na cópia do Drive, ou use **Arquivo > Fazer download > Fazer download do .ipynb**. A cópia do notebook não salva automaticamente os arquivos temporários do runtime. Localmente, compacte `resultados/` ou use as mesmas linhas sem importar `google.colab` e sem chamar `files.download`.
